<a href="https://colab.research.google.com/github/dibadabir/Signal-Analysis/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mne
!pip install mne-bids

import mne
import mne_bids
import pandas as pd
from pathlib import Path
import numpy as np
import logging
import warnings
import matplotlib.pyplot as plt

In [ ]:
# --- 1. CONFIGURATION ---

# **IMPORTANT: Set your BIDS root path here.**
BIDS_ROOT = Path("/content/drive/MyDrive/EEG/ds003490") # UPDATE THIS PATH

# Define processing parameters
L_FREQ = 0.5  # High-pass filter (Hz)
H_FREQ = 45.0 # Low-pass filter (Hz)
NOTCH = 50    # Notch filter (Hz)
# FIX: Changed TASK_NAME from "3AOB" to "Rest" based on the file system error
TASK_NAME = "Rest" # Auditory Oddball Task (the file name entity is 'Rest')

# Define channels of interest
CHANNELS_OF_INTEREST = [
    'F3', 'F4', 'Fz', 'FC1', 'FC2', 'FCz',
    'C3', 'C4', 'Cz', 'P3', 'P4', 'Pz',
    'Fp1', 'Fp2', 'T7', 'T8'
]

# Epoching parameters (20% pre-event, 80% post-event for a 1.5s total window)
EPOCH_WINDOW_LENGTH = 1.5 # seconds
TMIN = -0.2 * EPOCH_WINDOW_LENGTH # -0.3s
TMAX = 0.8 * EPOCH_WINDOW_LENGTH # 1.2s
# NOTE: S202 is a common 'Novelty' event code in oddball tasks.
# Check your BIDS events.tsv file to confirm the correct trigger code for the novelty/target events.
# Updated Event Dictionary
EVENT_ID = {
    'Standard Tone': 201,
    'Target Tone': 202,
    'Novel Tone': 203,
    'Eyes Open: Every 1000 ms/S  1': 1,
    'Eyes Open: Every 1000 ms/S  2': 2,
    'Eyes Closed: Every 1000 ms/S  3': 3,
    'Eyes Closed: Every 1000 ms/S  4': 4
}

# Standard 10-20 montage for electrode positioning
MONTAGE = mne.channels.make_standard_montage('standard_1020')

# Suppress specific MNE and BIDS warnings globally
mne.set_log_level('ERROR') # Only show errors, not warnings or info
warnings.filterwarnings("ignore", category=RuntimeWarning, module="mne_bids")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="mne")

# --- 2. SESSION SELECTION LOGIC ---

def get_session_info(bids_root):
    """Reads participants.tsv and selects the required sessions."""
    participants_path = bids_root / 'participants.tsv'
    if not participants_path.exists():
        raise FileNotFoundError(f"participants.tsv not found at {participants_path}")

    df = pd.read_csv(participants_path, sep='\t')
    selected_sessions = []

    for _, row in df.iterrows():
        subject_id = row['participant_id']
        group = row['Group']

        if group == 'CTL':
            # Control subjects: assume single session ('01')
            session = '01'
            session_label = 'n/a' # Placeholder for meds status
            selected_sessions.append((subject_id, session, session_label))

        elif group == 'PD':
            # PD subjects: find the 'OFF' medication session
            if row.get('sess1_Med', 'n/a') == 'OFF':
                session = '01'
                selected_sessions.append((subject_id, session, 'OFF'))
            elif row.get('sess2_Med', 'n/a') == 'OFF':
                session = '02'
                selected_sessions.append((subject_id, session, 'OFF'))

    return selected_sessions

# --- 3. CHANNEL FIXING (Specific to ds003490) ---

def fix_channel_types_for_ds003490(raw):
    """
    Manually assigns channel types as the ds003490 dataset uses 'n/a' in channels.tsv.
    """
    channel_types = {ch: "eeg" for ch in raw.ch_names}

    for ch in raw.ch_names:
        ch_up = ch.upper()
        if ch_up == "VEOG":
            channel_types[ch] = "eog"
        if ch_up in {"X", "Y", "Z"}:
            channel_types[ch] = "misc"

    raw.set_channel_types(channel_types)

    # Pick only the channels of interest for processing
    raw.pick_channels([ch for ch in CHANNELS_OF_INTEREST if ch in raw.ch_names])
    return raw

# --- 4. PROCESSING PIPELINE ---

def plot_preprocessing_results(raw_before, raw_after, ica, sub_id):
    """Generates plots to show the impact of filtering and ICA."""
    # 1. Plot PSD comparison (Filtering impact)
    fig, ax = plt.subplots(2, 1, figsize=(10, 8))
    fig.suptitle(f"Subject {sub_id}: Filtering Impact")
    raw_before.plot_psd(ax=ax[0], fmax=60, show=False)
    ax[0].set_title("Before (Raw Data)")
    raw_after.plot_psd(ax=ax[1], fmax=60, show=False)
    ax[1].set_title("After (0.5-45Hz + 50Hz Notch)")
    plt.tight_layout()
    plt.show()

    # 2. Plot ICA Overlay (Artifact removal impact)
    # This shows the signal with and without the excluded components
    ica.plot_overlay(raw_before, exclude=ica.exclude, picks='eeg', title=f"Subject {sub_id}: ICA Cleaning")
    plt.show()

def process_subject_data(bids_path):
    sub_id = bids_path.subject
    print(f"🚀 [{sub_id}] Starting pipeline...", end=" ", flush=True)

    try:
        raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False)
        raw.load_data()
        raw = fix_channel_types_for_ds003490(raw)

        # 1. IMMEDIATE FILTERING (To kill DC offsets/drifts)
        # We do this before re-referencing to keep errors from spreading
        raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, fir_design='firwin', verbose=False)
        raw.notch_filter(freqs=NOTCH, fir_design='firwin', verbose=False)
        print("➡️ Filtered", end=" ", flush=True)

        # 2. AUTOMATED BAD CHANNEL DETECTION
        # Drop channels that are flat (std dev near 0)
        channel_std = np.std(raw.get_data(), axis=1)
        flat_channels = [raw.ch_names[i] for i, std in enumerate(channel_std) if std < 1e-7]
        if flat_channels:
            raw.info['bads'].extend(flat_channels)
            print(f"➡️ Dropped Flat: {flat_channels}", end=" ", flush=True)
            raw.interpolate_bads(reset_bads=True)

        # 3. SET MONTAGE & REFERENCE
        raw.set_montage(MONTAGE, on_missing='ignore')
        raw.set_eeg_reference('average', projection=True, verbose=False)

        raw_before_ica = raw.copy() # For visualization comparison

        # 4. ROBUST ICA
        eog_channels = [ch for ch in ['Fp1', 'Fp2'] if ch in raw.ch_names]
        # ICA works better with a 1Hz high-pass filter
        ica_data = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)

        n_components = min(15, len(raw.ch_names) - len(raw.info['bads']) - 1)

        # Added 'infomax' and 'extended=True' as it's often better for noisy data
        ica = mne.preprocessing.ICA(n_components=n_components,
                                    random_state=97,
                                    method='infomax',
                                    fit_params=dict(extended=True))

        ica.fit(ica_data, verbose=False)

        # Automated EOG finding
        eog_indices, _ = ica.find_bads_eog(raw, ch_name=eog_channels, verbose=False)
        ica.exclude = eog_indices

        # Visual check (This should now look like real brain waves!)
        plot_preprocessing_results(raw_before_ica, raw.copy(), ica, sub_id)

        # Apply the cleaning
        ica.apply(raw, verbose=False)

        # 5. EPOCHING
        events, found_event_id = mne.events_from_annotations(raw, event_id=EVENT_ID, verbose=False)
        if len(events) == 0:
            print(f"⚠️ No events found for {sub_id}")
            return None

        epochs = mne.Epochs(raw, events, found_event_id, tmin=TMIN, tmax=TMAX,
                            baseline=(TMIN, 0), preload=True, verbose=False)

        # Reject truly bad trials
        epochs.drop_bad(reject=dict(eeg=250e-6), verbose=False)

        print(f"➡️ Done! (Kept {len(epochs)} epochs)")
        return epochs

    except Exception as e:
        print(f"\n❌ [{sub_id}] Failed: {e}")
        return None

# --- 5. MAIN EXECUTION ---

# Get the list of subjects and sessions to process
subjects_to_process = get_session_info(BIDS_ROOT)

all_epochs = []
for sub, ses, med_status in subjects_to_process:
    bids_path = mne_bids.BIDSPath(
        subject=sub.split('-')[1], # Remove 'sub-' prefix for MNE-BIDS
        session=ses,
        task=TASK_NAME,
        datatype='eeg',
        suffix='eeg',
        root=BIDS_ROOT
    )

    epochs = process_subject_data(bids_path)
    if epochs is not None:
        epochs.metadata = pd.DataFrame({'subject': sub, 'med_status': med_status}, index=epochs.selection)
        all_epochs.append(epochs)

    # --- DEBUGGING STEP ---
    # If the process_subject_data function returns None,
    # it means there was an issue loading or processing the data for this subject.
    # In such cases, we might want to inspect the raw annotations directly.
    if epochs is None:
        print(f"\nSkipping remaining processing due to error for {bids_path.subject}.")
        break # Stop processing after the first error to debug the event codes
    # --- END DEBUGGING STEP ---


# Concatenate all subjects' epochs into one MNE structure
if all_epochs:
    final_data = mne.concatenate_epochs(all_epochs)
    print("\n--- Processing Complete ---")
    print(f"Final data structure contains epochs from {len(final_data)} events.")
    print("The final 'final_data' object is an MNE Epochs object ready for feature extraction.")
else:
    print("\nNo data was successfully loaded or processed. Please check the BIDS_ROOT path and dataset structure.")